# Final Scientific Reports publication workflow

This notebook cold-executes the frozen Steps 2–10 workflow, writes every artifact to a unique Google Drive directory, and produces the checksummed final publication archive. It does not alter the Step 9 claim lock or refit the Step 8 law.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import shutil
import subprocess

REPO_URL = 'https://github.com/khalid-saqr/picoNewton.git'
PINNED_COMMIT = '__STEP10_IMPLEMENTATION_COMMIT__'
REPO_ROOT = Path('/content/picoNewton_step10')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + PINNED_COMMIT[:12]
DRIVE_ROOT = Path('/content/drive/MyDrive/picoNewton_susceptibility/final_runs') / RUN_ID
DRIVE_ROOT.mkdir(parents=True, exist_ok=False)

def run(command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run([str(item) for item in command], cwd=cwd, check=True)

print('Drive run root:', DRIVE_ROOT)

In [ ]:
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
run(['git', 'clone', REPO_URL, REPO_ROOT])
run(['git', 'checkout', '--detach', PINNED_COMMIT], cwd=REPO_ROOT)
resolved = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
assert resolved == PINNED_COMMIT
run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'])
run(['python', '-m', 'pip', 'install', '-e', REPO_ROOT / 'picoNewton_v3'])
run(['python', '-m', 'pip', 'install', '-e', f"{REPO_ROOT / 'piconewton_susceptibility'}[dev]"])

In [ ]:
ROOT = DRIVE_ROOT / 'workflow'
ROOT.mkdir(parents=True)
run(['piconewton-susceptibility-bootstrap', '--repo-root', REPO_ROOT, '--storage', 'local', '--local-root', ROOT])
run(['piconewton-susceptibility-step3', '--step2-root', ROOT/'bootstrap/step2', '--output', ROOT/'step3_parent_continuity', '--profile', 'publication'])
run(['piconewton-susceptibility-step4', '--step3-root', ROOT/'step3_parent_continuity', '--output', ROOT/'step4_perturbation', '--profile', 'publication'])
run(['piconewton-susceptibility-step5', '--step4-root', ROOT/'step4_perturbation', '--output', ROOT/'step5_harmonic_kernel', '--profile', 'publication'])
run(['piconewton-susceptibility-step6', '--step5-root', ROOT/'step5_harmonic_kernel', '--step4-root', ROOT/'step4_perturbation', '--output', ROOT/'step6_susceptibility', '--profile', 'publication'])
run(['piconewton-susceptibility-step7', '--step6-root', ROOT/'step6_susceptibility', '--output', ROOT/'step7_waveform_experiments', '--profile', 'publication'])
run(['piconewton-susceptibility-step8', '--step7-root', ROOT/'step7_waveform_experiments', '--output', ROOT/'step8_reduced_law', '--profile', 'publication'])
run(['piconewton-susceptibility-step9', '--step8-root', ROOT/'step8_reduced_law', '--output', ROOT/'step9_robustness_claim_lock', '--profile', 'publication'])

In [ ]:
FINAL = ROOT / 'step10_publication_archive'
run(['piconewton-susceptibility-step10', '--workflow-root', ROOT, '--repo-root', REPO_ROOT, '--output', FINAL, '--profile', 'publication'])
manifest = json.loads((FINAL/'step10_manifest.json').read_text())
assert manifest['workflow_complete'] is True
assert manifest['gates']['passed'] is True
expected = (FINAL/'publication_archive.sha256').read_text().split()[0]
digest = hashlib.sha256((FINAL/'publication_archive.zip').read_bytes()).hexdigest()
assert digest == expected == manifest['archive_sha256']
assert len(list((FINAL/'figures/main').glob('*.png'))) == 6
print(json.dumps(manifest, indent=2, sort_keys=True))

## Completion criterion

The workflow is complete only when the preceding cell prints a manifest with `workflow_complete: true`, verifies the archive SHA-256, and confirms all six main figures. The complete run remains in the unique Drive directory printed above.